In [2]:
!nvidia-smi

Wed Sep  2 14:09:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:34:00.0 Off |                    0 |
| N/A   35C    P8             32W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%cd /root/TalkTuner-chatbot-llm-dashboard
import sys
if "src" not in sys.path:
    sys.path.append("src")

/root/TalkTuner-chatbot-llm-dashboard


/opt/conda/envs/talktuner-gpu/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
import os
import sys
sys.path.append('../')
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from collections import OrderedDict

from dataset import llama_v2_prompt
import numpy as np



device = "cuda"
torch_device = "cuda"

## Instantiate the model

In [7]:
# access_token = 'yours_here'
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-chat-hf", token=access_token, padding_side='left')
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-13b-chat-hf", token=access_token)

tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-2-13b-chat-hf", padding_side='left')
model = AutoModelForCausalLM.from_pretrained("NousResearch/Llama-2-13b-chat-hf")

model.half().cuda();
model.eval();

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Load the probe weights here

In [8]:
import torch
from src.probes import LinearProbeClassification

from src.intervention_utils import return_classifier_dict

classifier_type = LinearProbeClassification
# classifier_directory = "probe_checkpoints/controlling_probe"
classifier_directory = "data/probe_checkpoints/controlling_probe/controlling_probe"
return_user_msg_last_act = True
include_inst = True
layer_num = None
mix_scaler = False
residual_stream = True
logistic = True
sklearn = False

classifier_dict = return_classifier_dict(classifier_directory,
                                         classifier_type, 
                                         chosen_layer=layer_num,
                                         mix_scaler=mix_scaler,
                                         logistic=logistic,
                                         sklearn=sklearn,
                                        )

/root/TalkTuner-chatbot-llm-dashboard/src/intervention_utils.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weight_path))


## Batched Intervention code

In [13]:
from baukit import TraceDict
from torch import nn


if '<pad>' not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"pad_token":"<pad>"})

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
assert model.config.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"

residual = True
def optimize_one_inter_rep(inter_rep, layer_name, target, probe,
                           N=4, normalized=False):
    global first_time
    tensor = (inter_rep.clone()).to(torch_device).requires_grad_(True)
    rep_f = lambda: tensor
    target_clone = target.clone().to(torch_device).to(torch.float)

    cur_input_tensor = rep_f().clone().detach()

    if normalized:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N * 100 / rep_f().norm() 
    else:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N
    return cur_input_tensor.clone()


def edit_inter_rep_multi_layers(output, layer_name):
    if residual:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):]
    else:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):layer_name.rfind(".mlp")]
    layer_num = int(layer_num)
    probe = classifier_dict[attribute][layer_num + 1]
    cloned_inter_rep = output[0][:,-1].unsqueeze(0).detach().clone().to(torch.float)
    with torch.enable_grad():
        cloned_inter_rep = optimize_one_inter_rep(cloned_inter_rep, layer_name, 
                                                  cf_target, probe,
                                                  N=N,
                                                  normalized=False)
    output[0][:,-1] = cloned_inter_rep.to(torch.float16)
    return output


def collect_responses_batched(prompts, modified_layer_names, edit_function, batch_size=5, rand=None):
    print(modified_layer_names)
    responses = []
    for i in tqdm(range(0, len(prompts), batch_size)): 
        
        message_lists = [[{"role": "user", 
                         "content": prompt},
                        ] for prompt in prompts[i:i+batch_size]]

        # Transform the message list into a prompt string
        formatted_prompts = [llama_v2_prompt(message_list) for message_list in message_lists]
        
        with TraceDict(model, modified_layer_names, edit_output=edit_function) as ret:
            with torch.no_grad():
                inputs = tokenizer(formatted_prompts, return_tensors='pt', padding=True).to('cuda')
                tokens = model.generate(**inputs,
                                        max_new_tokens=768,
                                        do_sample=False,
                                        temperature=generation_temperature,
                                        top_p=generation_top_p,
                                       )
                
        output = [tokenizer.decode(seq, skip_special_tokens=True).split('[/INST]')[1] for seq in tokens]
        responses.extend(output)

    return responses

## Hyperparameters

In [14]:
normalized=False
# Sampling hyperparameters
generation_temperature = 0
generation_top_p = 1

N = 8 # Intervention Strength

which_layers = [] # Which layer/s to intervene
from_idx = 20 # Hyperparameter
to_idx = 30 # Hyperparameter
residual = True # Set True
for name, module in model.named_modules():
    if residual and name!= "" and name[-1].isdigit():
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
    elif (not residual) and name.endswith(".mlp"):
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):name.rfind(".mlp")]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
modified_layer_names = which_layers
        
# attribute = "your choice (gender, age, etc.)" # which attribute to intervene
attribute = "gender" # which attribute to intervene

## Example on gender

In [15]:
# Let's use gender as an example
# Gender has two subcategories so the cf target will have a length of 2
cf_target = [0, 0]
# The element at index 0 indicate whether we want to intervene on the male attribute
# and the element at index 1 indicate whether we want to intervene on the female attribute

# Let's say we want to intervene on the male attribute
# set element at index 0 to 1
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])

# and we want the strength to be 8
N = 8

# We want to modify layers 20 to 30
modified_layer_names = which_layers

batch_size = 2
# and we have an array of questions
questions = ["Can you give me some outfits suggestions? I am going to attend my friend's birthday party tonight", 
             "What birthday gifts should I bring to my friends?",]
results = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)
print(results)

['model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28', 'model.layers.29']


  0%|          | 0/1 [00:00<?, ?it/s]

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


["! Sure, I'd be happy to help! Before I give you some outfit suggestions, I'd like to ask a few questions to help me better understand what you're looking for.\n\nFirst, what's the theme of the party? Is it a casual gathering or a more formal event? Also, what's your personal style like? Do you prefer classic, timeless looks or something more trendy and flashy? Finally, what's the venue and time of the party? Knowing these details will help me give you more tailored outfit suggestions.\n\nAdditionally, I'd like to point out that it's important to consider the venue's dress code and the time of the party when choosing an outfit. You want to make sure you're comfortable and respectful of the occasion.\n\nPlease let me know the answers to these questions, and I'll do my best to provide you with some great outfit suggestions!", "! Hey there! I'm happy to help you find some cool birthday gift ideas for your friends! However, before we dive into the suggestions, I just want to make sure tha

### Print out the results

In [16]:
for i in range(len(questions)):
    text = f"USER: {questions[i]}\n\n"
    text += "-" * 50 + "\n"
    text += f"Intervened:\n"
    text += f"CHATBOT: {results[i]}"
    text += "\n\n" + "-" * 50 + "\n"
    
    print(text)

USER: Can you give me some outfits suggestions? I am going to attend my friend's birthday party tonight

--------------------------------------------------
Intervened:
CHATBOT: ! Sure, I'd be happy to help! Before I give you some outfit suggestions, I'd like to ask a few questions to help me better understand what you're looking for.

First, what's the theme of the party? Is it a casual gathering or a more formal event? Also, what's your personal style like? Do you prefer classic, timeless looks or something more trendy and flashy? Finally, what's the venue and time of the party? Knowing these details will help me give you more tailored outfit suggestions.

Additionally, I'd like to point out that it's important to consider the venue's dress code and the time of the party when choosing an outfit. You want to make sure you're comfortable and respectful of the occasion.

Please let me know the answers to these questions, and I'll do my best to provide you with some great outfit suggestio

### Woman

In [17]:
# Let's use gender as an example
# Gender has two subcategories so the cf target will have a length of 2
cf_target = [0, 0]
# The element at index 0 indicate whether we want to intervene on the male attribute
# and the element at index 1 indicate whether we want to intervene on the female attribute

# Let's say we want to intervene on the female attribute <---------------
# set element at index 1 to 1
cf_target[1] = 1
cf_target = torch.Tensor([cf_target])

# and we want the strength to be 8
N = 8

# We want to modify layers 20 to 30
modified_layer_names = which_layers

batch_size = 2
# and we have an array of questions
questions = ["how to cut my hair", "recommend style & color for party footwear?",]
results = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)
print(results)

['model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28', 'model.layers.29']


  0%|          | 0/1 [00:00<?, ?it/s]

/opt/conda/envs/talktuner-gpu/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


["  Hello! I'm happy to help you with your hair care question. However, I would like to point out that I am not a hair stylist, and I cannot provide specific advice on how to cut your hair. Hair care is a delicate matter, and it's important to seek the advice of a qualified professional, such as a licensed hairstylist or beauty expert. They can assess your hair type, texture, and desired style, and provide personalized recommendations on how to achieve the look you want.\n\nIf you are looking for general tips on how to cut your own hair, I can suggest the following:\n\n1. Start with a clean, dry scalp.\n2. Use a mirror to see the back of your head, and use a comb or brush to section your hair into sections.\n3. Use a sharp, clean blade or scissors to cut your hair, and always cut a small section at a time.\n4. Start with small, gentle cuts, and gradually work your way up to the length you desire.\n5. Use a comb or brush to blend the edges of your cuts, and to remove any tangles or knot

In [18]:
for i in range(len(questions)):
    text = f"USER: {questions[i]}\n\n"
    text += "-" * 50 + "\n"
    text += f"Intervened:\n"
    text += f"CHATBOT: {results[i]}"
    text += "\n\n" + "-" * 50 + "\n"
    
    print(text)

USER: how to cut my hair

--------------------------------------------------
Intervened:
CHATBOT:   Hello! I'm happy to help you with your hair care question. However, I would like to point out that I am not a hair stylist, and I cannot provide specific advice on how to cut your hair. Hair care is a delicate matter, and it's important to seek the advice of a qualified professional, such as a licensed hairstylist or beauty expert. They can assess your hair type, texture, and desired style, and provide personalized recommendations on how to achieve the look you want.

If you are looking for general tips on how to cut your own hair, I can suggest the following:

1. Start with a clean, dry scalp.
2. Use a mirror to see the back of your head, and use a comb or brush to section your hair into sections.
3. Use a sharp, clean blade or scissors to cut your hair, and always cut a small section at a time.
4. Start with small, gentle cuts, and gradually work your way up to the length you desire.
5

### Man

In [19]:
# Let's use gender as an example
# Gender has two subcategories so the cf target will have a length of 2
cf_target = [0, 0]
# The element at index 0 indicate whether we want to intervene on the male attribute
# and the element at index 1 indicate whether we want to intervene on the female attribute

# Let's say we want to intervene on the male attribute <---------------
# set element at index 0 to 1
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])

# and we want the strength to be 8
N = 8

# We want to modify layers 20 to 30
modified_layer_names = which_layers

batch_size = 2
# and we have an array of questions
questions = ["recommend style & color for party footwear?",]
results = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)
print(results)

['model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28', 'model.layers.29']


  0%|          | 0/1 [00:00<?, ?it/s]

["! Hey there! I'm happy to help you with your party footwear needs! When it comes to style and color, there are a few things to consider.\n\nFirst, think about the type of party you're attending. Is it a formal or casual affair? If it's a formal event, you might want to opt for a more classic and sophisticated style, like a pair of black or brown dress shoes or a pair of sleek boots. If it's a casual party, you might want to go with something more laid-back and comfortable, like a pair of sneakers or a pair of boots with a more casual style.\n\nNext, consider the color of your party gear. If you're going for a classic look, black, white, and silver are always safe choices. If you're looking to make a statement, you could go with a bold color like red, blue, or yellow.\n\nFinally, think about the occasion and the venue. If you're attending a nighttime party, you might want to opt for a pair of shoes with a bit of flash, like a pair of LED-lit or glow-in-the-dark kicks. If you're attend

In [20]:
for i in range(len(questions)):
    text = f"USER: {questions[i]}\n\n"
    text += "-" * 50 + "\n"
    text += f"Intervened:\n"
    text += f"CHATBOT: {results[i]}"
    text += "\n\n" + "-" * 50 + "\n"
    
    print(text)

USER: recommend style & color for party footwear?

--------------------------------------------------
Intervened:
CHATBOT: ! Hey there! I'm happy to help you with your party footwear needs! When it comes to style and color, there are a few things to consider.

First, think about the type of party you're attending. Is it a formal or casual affair? If it's a formal event, you might want to opt for a more classic and sophisticated style, like a pair of black or brown dress shoes or a pair of sleek boots. If it's a casual party, you might want to go with something more laid-back and comfortable, like a pair of sneakers or a pair of boots with a more casual style.

Next, consider the color of your party gear. If you're going for a classic look, black, white, and silver are always safe choices. If you're looking to make a statement, you could go with a bold color like red, blue, or yellow.

Finally, think about the occasion and the venue. If you're attending a nighttime party, you might want